# Lab 2 · Tokens and chat templates

**Time:** 20 min · **You will:** see text the way the model sees it — as token IDs — and find out why that matters for cost, context length and chat formatting.

A language model never sees letters or words. A **tokenizer** cuts text into pieces (tokens) from a fixed vocabulary and turns each piece into an integer ID. Everything the model reads and writes is a sequence of these IDs.

*How to use this notebook: **Kernel → Restart Kernel and Run All Cells**, read each result, then change the `🔧` values and run those cells again (Shift+Enter).*

In [ ]:
import sys, os
# find the course folder (the one holding slm_lab.py), whichever folder this notebook was opened from
sys.path.insert(0, next(os.path.abspath(d) for d in ("..", "../..") if os.path.exists(os.path.join(d, "slm_lab.py"))))
from slm_lab import *
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL_ID)
print(f"Vocabulary size: {len(tok):,} tokens")

## 2.1 Text → tokens → IDs
🔧 **Try:** change `text` to your own sentence, e.g. your name or a sentence with a rare word.

In [ ]:
text = "Employees are entitled to 20 days of paternity leave."   # 🔧 type any sentence
tokens = tok.tokenize(text)
ids = tok.encode(text)
for t, i in zip(tokens, ids):
    print(f"{i:>7}  {t!r}")

`Ġ` marks a leading space: `'Ġleave'` and `'leave'` are **different tokens**. Common words are one token; rarer ones (*paternity*) are split into pieces.

### 🔧 Try it 2.1 — Which language costs more?
Models are billed and limited by **tokens**, not words. The cell counts the tokens in the same sentence in English and Hindi, and in a line of code.

🔧 **Try:** add a line for another language you know, e.g. `"tamil": "…",` — or change a sentence.

In [ ]:
samples = {                                                          # 🔧 change or add sentences
    "english": "Employees get 20 days of paternity leave.",
    "hindi": "कर्मचारियों को 20 दिन का पितृत्व अवकाश मिलता है।",
    "code": "if leave_days > 20: raise ValueError('too many days')",
}

counts = {name: len(tok.encode(s)) for name, s in samples.items()}
for name, n in counts.items():
    print(f"{name:8} {n:3} tokens  ({len(samples[name].split())} words)")

In [ ]:
check("english" in counts and "hindi" in counts,
      f"Hindi needs {counts.get('hindi', 0) / max(counts.get('english', 1), 1):.1f}× the tokens of English for the "
      "same meaning - it costs more and fills the context window faster.",
      "keep the 'english' and 'hindi' lines in samples to compare them.")

**Why?** The vocabulary was learned mostly from English text and code, so English words get whole tokens while Devanagari is split into many small pieces. When you pick or train a model for Indian languages, check its tokenizer first.

### 🔧 Try it 2.2 — Back from IDs to text
`tok.decode` turns IDs back into text. The cell decodes a list of IDs and finds the ID of the word `" leave"` (with its leading space).

🔧 **Try:** change `word` to `"leave"` (no space) or `" holiday"` — a different spelling gives a different ID.

In [ ]:
ids_to_read = [50, 865, 5274, 374, 220, 16, 17, 2849, 264, 1042, 13]   # 🔧 change the numbers
word = " leave"                                                     # 🔧 any word

decoded = tok.decode(ids_to_read)
word_ids = tok.encode(word)
print("decoded :", repr(decoded))
print(f"{word!r} → {word_ids}  ({len(word_ids)} token{'s' if len(word_ids) > 1 else ''})")

In [ ]:
check(len(decoded) > 0 and len(word_ids) > 0,
      f"IDs → text works both ways. {word!r} is {len(word_ids)} token(s); with or without the space it's a "
      "different token.", "decoding failed - check that ids_to_read is a list of numbers.")

## 2.2 Chat templates: how a conversation becomes one long text
A chat model is still a next-token predictor. The **chat template** wraps each message in special tokens so the model knows who is speaking and when it's its turn.

🔧 **Try:** change the user's question and run the cell again.

In [ ]:
messages = [
    {"role": "system", "content": "You are ABC Company's HR assistant."},
    {"role": "user", "content": "How many sick days do I get?"},          # 🔧 change the question
]
prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(prompt)

### 👀 See it 2.3 — Read the template
The cell lists the **special tokens** in the prompt above, and shows what `add_generation_prompt=True` adds at the end (by building the prompt again without it and comparing).

In [ ]:
import re
special = sorted(set(re.findall(r"<\|[a-z_]+\|>", prompt)))
without = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
gen_prompt = prompt[len(without):]
print("special tokens       :", special)
print("generation prompt    :", repr(gen_prompt))

In [ ]:
check(set(special) == {"<|im_start|>", "<|im_end|>"} and gen_prompt.strip() == "<|im_start|>assistant",
      "The template marks each turn with <|im_start|>role … <|im_end|>, and the generation prompt opens the "
      "assistant's turn so the model's next tokens are its reply.",
      "the messages above should have a system and a user message.")

⚠️ **Lesson for fine-tuning (Extra Lab B):** train and use a model with the *same* template. If you fine-tune on one format and chat with another, quality drops sharply.

## 2.3 Does the whole HR policy fit in the prompt?
Every model has a **context window**: the maximum number of tokens it can attend to at once (prompt + answer).

### 👀 See it 2.4
The cell counts the tokens in the full HR policy and reads the model's context window from its configuration.

In [ ]:
from transformers import AutoConfig
config = AutoConfig.from_pretrained(MODEL_ID)
policy_tokens = len(tok.encode(policy_text()))
context_window = config.max_position_embeddings
print(f"Policy: {policy_tokens:,} tokens · context window: {context_window:,} tokens "
      f"→ uses {policy_tokens / context_window:.1%}")

In [ ]:
check(policy_tokens < context_window,
      "This short policy fits easily. A real company handbook (hundreds of pages) would not - and every token in "
      "the prompt costs compute on every question. Lab 6 retrieves only the relevant section instead.",
      "the policy is longer than the context window.")

## 🧠 Check your understanding
1. Why can the same sentence cost a different amount in two languages?
2. What would go wrong if you fine-tuned with one chat template and served the model with another?

*✍️ Your answer:* 

**Next:** Lab 3 — why Transformers beat RNNs and LSTMs.